# NumPy: Math, Statistics & Linear Algebra

This notebook is where NumPy starts feeling powerful. We're done with just creating arrays — now we're going to do real math on them.

**Stuff covered here:**
1. Universal functions (ufuncs) — element-wise math on whole arrays
2. Aggregations and axes — sum, mean, max, min, std, var
3. Statistical functions — median, percentile, correlation
4. Linear algebra basics — matrix multiplication (dot product)
5. Solving linear systems
6. Determinants, inverses, eigenvalues

A lot of this is the math behind machine learning. Don't worry if some terms feel new — I'll define them as we go.

In [1]:
import numpy as np
np.random.seed(42)
print("NumPy version:", np.__version__)

NumPy version: 1.26.4


## Universal Functions (ufuncs)

**ufunc** = universal function. It's a function that runs on every single element of an array at once, no for-loop needed. So instead of looping over a list to take square roots one by one, you write `np.sqrt(arr)` and NumPy handles all of them in one shot — and it's way faster because it runs in C under the hood.

You've already been using them (`+`, `*`, `np.sin`, etc.). Here's a tour of the most useful ones.

### Basic math

In [2]:
a = np.array([1, 4, 9, 16, 25])

print("Square root:", np.sqrt(a))
print("Squared:    ", np.square(a))
print("Exp:        ", np.exp(np.array([0, 1, 2])))
print("Log:        ", np.log(np.array([1, np.e, np.e**2])))
print("Log10:      ", np.log10(np.array([1, 10, 100, 1000])))

Square root: [1. 2. 3. 4. 5.]
Squared:     [  1  16  81 256 625]
Exp:         [1.         2.71828183 7.3890561 ]
Log:         [0. 1. 2.]
Log10:       [0. 1. 2. 3.]


### Trigonometry

NumPy's trig functions expect **radians**, not degrees. If you've got degrees, convert first with `np.deg2rad()`. Forgetting this is a classic "why is my answer wrong" moment.

In [3]:
# Angles in radians
angles = np.array([0, np.pi/4, np.pi/2, np.pi])

print("Angles (radians):", angles)
print("Sin:             ", np.sin(angles).round(4))
print("Cos:             ", np.cos(angles).round(4))
print("Tan:             ", np.tan(angles).round(4))

# Converting between degrees and radians
degrees = np.array([0, 30, 45, 60, 90])
rad = np.deg2rad(degrees)
print("\nDegrees:", degrees)
print("Radians:", rad.round(4))

Angles (radians): [0.         0.78539816 1.57079633 3.14159265]
Sin:              [0.     0.7071 1.     0.    ]
Cos:              [ 1.      0.7071  0.     -1.    ]
Tan:              [ 0.00000000e+00  1.00000000e+00  1.63312394e+16 -0.00000000e+00]

Degrees: [ 0 30 45 60 90]
Radians: [0.     0.5236 0.7854 1.0472 1.5708]


### Rounding and integer operations

A few rounding flavors that confuse people:
- **`floor`** — round *down* toward negative infinity (so `-1.6` becomes `-2`)
- **`ceil`** — round *up* toward positive infinity
- **`trunc`** — chop off the decimals, always heading toward zero (so `-1.6` becomes `-1`, not `-2`)
- **`round`** — standard rounding to nearest, with .5 going to even (banker's rounding)

In [4]:
x = np.array([1.234, 2.5, 3.7, -0.5, -1.6])

print("Round:    ", np.round(x))
print("Floor:    ", np.floor(x))   # round DOWN
print("Ceil:     ", np.ceil(x))    # round UP
print("Truncate: ", np.trunc(x))   # toward zero
print("Absolute: ", np.abs(x))

Round:     [ 1.  2.  4. -0. -2.]
Floor:     [ 1.  2.  3. -1. -2.]
Ceil:      [ 2.  3.  4. -0. -1.]
Truncate:  [ 1.  2.  3. -0. -1.]
Absolute:  [1.234 2.5   3.7   0.5   1.6  ]


### Element-wise arithmetic on two arrays

In [5]:
a = np.array([10, 20, 30, 40])
b = np.array([1, 2, 3, 4])

print("a + b =", a + b)
print("a - b =", a - b)
print("a * b =", a * b)
print("a / b =", a / b)
print("a // b =", a // b)   # integer division
print("a % b =", a % b)     # modulo
print("a ** b =", a ** b)   # power

# Equivalent ufunc form
print("\nnp.add(a, b)      =", np.add(a, b))
print("np.multiply(a, b) =", np.multiply(a, b))

a + b = [11 22 33 44]
a - b = [ 9 18 27 36]
a * b = [ 10  40  90 160]
a / b = [10. 10. 10. 10.]
a // b = [10 10 10 10]
a % b = [0 0 0 0]
a ** b = [     10     400   27000 2560000]

np.add(a, b)      = [11 22 33 44]
np.multiply(a, b) = [ 10  40  90 160]


## Aggregations — Reducing Arrays to Numbers

An **aggregation** is anything that takes a bunch of values and squashes them into fewer numbers — like turning 100 grades into a single average. `sum`, `mean`, `max`, `min`, `std` — all aggregations.

In 1D this is boring (just one answer). The interesting part shows up in 2D+, where you have to decide which direction to collapse. That direction is called the **axis**.

In [6]:
# 1D — straightforward
a = np.array([3, 7, 2, 8, 5, 1, 9, 4])

print(f"Sum:      {a.sum()}")
print(f"Mean:     {a.mean()}")
print(f"Min:      {a.min()}")
print(f"Max:      {a.max()}")
print(f"Argmin:   {a.argmin()}    (index of minimum)")
print(f"Argmax:   {a.argmax()}    (index of maximum)")
print(f"Std:      {a.std():.4f}")
print(f"Variance: {a.var():.4f}")
print(f"Product:  {a.prod()}")

Sum:      39
Mean:     4.875
Min:      1
Max:      9
Argmin:   5    (index of minimum)
Argmax:   6    (index of maximum)
Std:      2.7128
Variance: 7.3594
Product:  60480


### The `axis` argument — the most important concept here

The **axis** is which dimension you're collapsing. For a 2D array of shape `(rows, cols)`:
- `axis=0` → collapse DOWN the rows → you get one value per **column**
- `axis=1` → collapse ACROSS the columns → you get one value per **row**

**The trick I use to remember:** the axis you pass is the one that **disappears** from the shape. Pass `axis=0` on a `(3, 4)` array → result is shape `(4,)`. The 3 is gone.



In [7]:
m = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])
print("Matrix (3 rows, 4 cols):")
print(m)

print(f"\nm.shape = {m.shape}")
print(f"\nm.sum()         = {m.sum()}        ← scalar, all elements")
print(f"m.sum(axis=0)   = {m.sum(axis=0)}   ← shape (4,), summed down each col")
print(f"m.sum(axis=1)   = {m.sum(axis=1)}      ← shape (3,), summed across each row")

Matrix (3 rows, 4 cols):
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

m.shape = (3, 4)

m.sum()         = 78        ← scalar, all elements
m.sum(axis=0)   = [15 18 21 24]   ← shape (4,), summed down each col
m.sum(axis=1)   = [10 26 42]      ← shape (3,), summed across each row


In [8]:
# Same idea for mean, max, min, etc.
m = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])

print("Column means:", m.mean(axis=0))
print("Row means:   ", m.mean(axis=1))
print()
print("Column max: ", m.max(axis=0))
print("Row max:    ", m.max(axis=1))

Column means: [5. 6. 7. 8.]
Row means:    [ 2.5  6.5 10.5]

Column max:  [ 9 10 11 12]
Row max:     [ 4  8 12]


### `keepdims` — preserve the reduced axis as size 1

By default, the axis you reduce disappears. **`keepdims=True`** tells NumPy to keep that axis around as size 1 instead of dropping it.

Why bother? Because when you want to do something like "subtract the row mean from each row," the shapes need to line up for broadcasting. Without `keepdims`, the means come out as `(3,)` and the math gets ugly. With `keepdims`, they come out as `(3, 1)` and broadcasting just works.

In [9]:
m = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])

# Without keepdims — result is 1D
means_1 = m.mean(axis=1)
print("Without keepdims:")
print(f"  shape: {means_1.shape}")
print(f"  values: {means_1}")

# With keepdims — result keeps the dim
means_2 = m.mean(axis=1, keepdims=True)
print("\nWith keepdims:")
print(f"  shape: {means_2.shape}")
print(means_2)

# Now you can subtract directly (broadcasting works cleanly)
centered = m - means_2
print("\nMean-centered:")
print(centered)

Without keepdims:
  shape: (3,)
  values: [ 2.5  6.5 10.5]

With keepdims:
  shape: (3, 1)
[[ 2.5]
 [ 6.5]
 [10.5]]

Mean-centered:
[[-1.5 -0.5  0.5  1.5]
 [-1.5 -0.5  0.5  1.5]
 [-1.5 -0.5  0.5  1.5]]


## Statistical Functions

NumPy has statistics built in — handy for quick analysis without reaching for pandas.

Quick glossary so the output below makes sense:

- **Mean** — the average (sum / count).
- **Median** — the middle value once you sort the data. Less affected by outliers than the mean.
- **Standard deviation (std)** — how spread out the values are from the mean. Small = values are tightly bunched, large = scattered.
- **Variance** — standard deviation squared. Same info, different units. (Variance is in "units²", std is in the original units, which is why people usually quote std.)
- **Percentile** — the value below which a given % of the data falls. The 25th percentile is the value that 25% of the data sits below.
- **IQR (Interquartile Range)** — 75th percentile minus 25th percentile. A robust way to talk about spread that ignores extreme outliers.

In [10]:
data = np.array([85, 92, 78, 96, 88, 71, 84, 90, 79, 95, 87, 82])

print(f"Mean:           {data.mean():.2f}")
print(f"Median:         {np.median(data):.2f}")
print(f"Std dev:        {data.std():.2f}")
print(f"Variance:       {data.var():.2f}")
print(f"Min:            {data.min()}")
print(f"Max:            {data.max()}")
print(f"Range:          {data.max() - data.min()}")
print(f"25th percentile: {np.percentile(data, 25):.2f}")
print(f"75th percentile: {np.percentile(data, 75):.2f}")
print(f"IQR:            {np.percentile(data, 75) - np.percentile(data, 25):.2f}")

Mean:           85.58
Median:         86.00
Std dev:        7.04
Variance:       49.58
Min:            71
Max:            96
Range:          25
25th percentile: 81.25
75th percentile: 90.50
IQR:            9.25


### Correlation and covariance — quick definitions

- **Covariance** — tells you whether two variables move together. Positive = when one goes up, the other tends to go up too. Negative = they move opposite. The number itself is hard to interpret because it depends on the units.
- **Correlation coefficient** — basically covariance scaled to live between -1 and 1. Way easier to read: `+1` is a perfect upward line, `-1` is a perfect downward line, `0` means no linear relationship.

In short: correlation is the "normalized" version of covariance you actually want to look at.

In [11]:
# Correlation and covariance
hours_studied = np.array([2, 4, 6, 8, 10, 12])
exam_scores   = np.array([50, 60, 70, 75, 85, 95])

# Correlation coefficient (Pearson)
corr_matrix = np.corrcoef(hours_studied, exam_scores)
print("Correlation matrix:")
print(corr_matrix)
print(f"\nCorrelation: {corr_matrix[0, 1]:.4f}  (near 1.0 = strong positive)")

# Covariance
cov = np.cov(hours_studied, exam_scores)
print(f"\nCovariance: {cov[0, 1]:.4f}")

Correlation matrix:
[[1.         0.99679058]
 [0.99679058 1.        ]]

Correlation: 0.9968  (near 1.0 = strong positive)

Covariance: 61.0000


In [12]:
# Unique values, counts, sorting
data = np.array([3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5])

print("Unique values:", np.unique(data))

# With counts
vals, counts = np.unique(data, return_counts=True)
for v, c in zip(vals, counts):
    print(f"  {v} appears {c} times")

print("\nSorted ascending: ", np.sort(data))
print("Sorted descending:", np.sort(data)[::-1])

Unique values: [1 2 3 4 5 6 9]
  1 appears 2 times
  2 appears 1 times
  3 appears 2 times
  4 appears 1 times
  5 appears 3 times
  6 appears 1 times
  9 appears 1 times

Sorted ascending:  [1 1 2 3 3 4 5 5 5 6 9]
Sorted descending: [9 6 5 5 5 4 3 3 2 1 1]


## Linear Algebra — Matrix Multiplication

This is the part where NumPy starts feeling like the engine behind machine learning. Linear regression, neural networks, PCA — all of it is matrix multiplication underneath.

A few definitions before we go:

- **Vector** — a 1D array of numbers. Like `[1, 2, 3]`.
- **Matrix** — a 2D grid of numbers. Rows and columns.
- **Dot product** — for two vectors of the same length, multiply them element-wise and add it all up. One scalar comes out.

```
a = [a₁, a₂, a₃]
b = [b₁, b₂, b₃]
a · b = a₁b₁ + a₂b₂ + a₃b₃
```

Dot product tells you how aligned two vectors are. If they point the same way, it's big and positive. If they're perpendicular, it's zero. Useful point for later.

In [13]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Manual: 1*4 + 2*5 + 3*6 = 4 + 10 + 18 = 32
print("Manual dot:  ", 1*4 + 2*5 + 3*6)
print("np.dot(a, b):", np.dot(a, b))
print("a @ b:       ", a @ b)        # @ is the matmul operator

Manual dot:   32
np.dot(a, b): 32
a @ b:        32


If you are doing deep learning, computer vision, or standard 2D/3D matrix math, always default to @ (np.matmul). It behaves exactly like standard mathematical matrix batches and is much less prone to creating unexpected, giant dimensional arrays.

### Matrix multiplication

**Matrix multiplication** generalizes the dot product to whole matrices. For matrices `A` (shape `m × n`) and `B` (shape `n × p`), the product `A @ B` has shape `m × p`.

The middle dimensions have to match: the columns of `A` must equal the rows of `B`. If they don't, NumPy throws a shape error and you go fix it.

Mental model: each entry of the result is a dot product of one row of `A` with one column of `B`.

In [14]:
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])           # 3x2

B = np.array([[7, 8, 9],
              [10, 11, 12]])     # 2x3

C = A @ B                        # 3x3
print(f"A shape: {A.shape}")
print(f"B shape: {B.shape}")
print(f"\nA @ B = (shape {C.shape}):")
print(C)

A shape: (3, 2)
B shape: (2, 3)

A @ B = (shape (3, 3)):
[[ 27  30  33]
 [ 61  68  75]
 [ 95 106 117]]


### `*` vs `@` — two completely different operations

This trips up basically everyone at first. They look similar but they do totally different things.

- **`*`** — element-wise multiplication (also called the **Hadamard product**). Multiply matching positions. Shapes have to match (or be broadcastable).
- **`@`** — actual matrix multiplication. Row-by-column dot products. Shapes have to follow the matmul rule.

In [15]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

# Element-wise multiplication (Hadamard product)
print("A * B (element-wise):")
print(A * B)

# Matrix multiplication
print("\nA @ B (matrix multiply):")
print(A @ B)
# A @ B[0,0] = 1*5 + 2*7 = 19
# A @ B[0,1] = 1*6 + 2*8 = 22
# A @ B[1,0] = 3*5 + 4*7 = 43
# A @ B[1,1] = 3*6 + 4*8 = 50

A * B (element-wise):
[[ 5 12]
 [21 32]]

A @ B (matrix multiply):
[[19 22]
 [43 50]]


**Rule of thumb:** `*` is element-wise. `@` is matrix multiplication. If your output shape looks wrong, this is the first thing to check — confusing the two is a classic ML bug.

## More Linear Algebra (`np.linalg`)

The `numpy.linalg` submodule is where the heavier linear algebra lives. Quick rundown of the terms in the next few cells:

- **Determinant** — a single number computed from a square matrix. The most useful thing about it: if `det = 0`, the matrix is *singular* and can't be inverted. Geometrically, it's how much a transformation scales area (in 2D) or volume (in 3D).
- **Inverse** — the "undo" matrix. If `A @ A_inv` gives you the identity matrix (1s on the diagonal, 0s elsewhere), then `A_inv` is the inverse of `A`. Not every matrix has one.
- **Identity matrix** — the matrix equivalent of the number 1. Multiplying by it leaves things unchanged.

In [16]:
A = np.array([[4, 7],
              [2, 6]])

# Determinant
det = np.linalg.det(A)
print(f"Determinant: {det:.4f}")

# Inverse
inv = np.linalg.inv(A)
print("\nInverse:")
print(inv)

# Verify: A @ A_inv should equal the identity
identity = A @ inv
print("\nA @ A^-1 (should be identity):")
print(identity.round(6))

Determinant: 10.0000

Inverse:
[[ 0.6 -0.7]
 [-0.2  0.4]]

A @ A^-1 (should be identity):
[[ 1.  0.]
 [-0.  1.]]


### Solving systems of linear equations

A **linear system** is just a bunch of equations stacked together that all need to be satisfied at once. You can write it compactly as `Ax = b`, where `A` is the matrix of coefficients, `x` is the unknown vector, and `b` is the right-hand side.

Solving it means finding the values of `x` that make every equation true. `np.linalg.solve(A, b)` does this for you.

Example — find the `x` and `y` that satisfy both:
```
2x + 3y = 13
x  - y  = 1
```

In [17]:
# Coefficients matrix
A = np.array([[2, 3],
              [1, -1]])

# Right-hand side
b = np.array([13, 1])

# Solve
x = np.linalg.solve(A, b)
print(f"x = {x[0]:.2f}, y = {x[1]:.2f}")

# Verify
print("\nCheck: A @ x =", A @ x, "(should equal b =", b, ")")

x = 3.20, y = 2.20

Check: A @ x = [13.  1.] (should equal b = [13  1] )


**Tip:** `np.linalg.solve(A, b)` is faster and more numerically stable than `np.linalg.inv(A) @ b`. Computing the inverse explicitly is almost always a bad idea — it accumulates more floating-point error. So if you ever see yourself writing `inv(A) @ b`, swap it for `solve`.

## Eigenvalues and Eigenvectors (briefly)

- **Eigenvector** — a special vector that, when you multiply it by a matrix, doesn't change direction. The matrix just *stretches* or *shrinks* it. No rotation, no skew.
- **Eigenvalue (λ)** — the scalar that tells you by how much the eigenvector got stretched.

In one equation: `A @ v = λ * v`. The matrix `A` acts on `v` like simple scalar multiplication.

Why anyone cares: PCA, spectral clustering, recommender systems, Google's original PageRank — they all rely on eigenvalues / eigenvectors. They're how you find the "natural directions" hidden in a matrix.

In [18]:
A = np.array([[4, 1],
              [2, 3]])

eigenvalues, eigenvectors = np.linalg.eig(A)

print("Eigenvalues:", eigenvalues)
print("\nEigenvectors (each column is an eigenvector):")
print(eigenvectors)

# Verify: A @ v ≈ λ * v
v0 = eigenvectors[:, 0]
print(f"\nA @ v0  = {A @ v0}")
print(f"λ0 * v0 = {eigenvalues[0] * v0}")

Eigenvalues: [5. 2.]

Eigenvectors (each column is an eigenvector):
[[ 0.70710678 -0.4472136 ]
 [ 0.70710678  0.89442719]]

A @ v0  = [3.53553391 3.53553391]
λ0 * v0 = [3.53553391 3.53553391]


## Practice Exercises

### Exercise 1
Given `arr = np.array([4, 9, 16, 25, 36])`, compute the square root, natural log, and the sum of all those square roots.

### Exercise 2
Create a 4×3 matrix of random integers between 1 and 100. Compute:
- The sum of each column
- The mean of each row
- The maximum value of the whole matrix and its position (use argmax + unravel_index)

### Exercise 3
You have height (cm) and weight (kg) data for 5 people:
```
heights = np.array([165, 172, 180, 158, 175])
weights = np.array([60, 72, 85, 50, 78])
```
- Compute the BMI for each person: `weight / (height_in_meters)²`
- Find the correlation coefficient between height and weight

### Exercise 4 (linear algebra)
Solve this system of equations:
```
3x + 2y - z = 1
2x - 2y + 4z = -2
-x + 0.5y - z = 0
```

### Exercise 5 (challenge)
Use the Normal Equation to fit a line to these points:
```
x = np.array([1, 2, 3, 4, 5, 6, 7])
y = np.array([2.1, 4.0, 6.1, 7.9, 10.2, 11.8, 14.1])
```
Report the slope and intercept.

In [21]:
# Exercise 1
arr = np.array([4, 9, 16, 25, 36])
sqrts = np.sqrt(arr)
print("Square roots:", sqrts)
print("Natural logs:", np.log(arr))
print("Sum of sqrts:", sqrts.sum())

Square roots: [2. 3. 4. 5. 6.]
Natural logs: [1.38629436 2.19722458 2.77258872 3.21887582 3.58351894]
Sum of sqrts: 20.0


In [22]:
# Exercise 2
np.random.seed(1)
m = np.random.randint(1, 101, size=(4, 3))
print("Matrix:")
print(m)
print(f"\nColumn sums: {m.sum(axis=0)}")
print(f"Row means:    {m.mean(axis=1)}")
print(f"Max value:    {m.max()}")

flat_idx = m.argmax()
position = np.unravel_index(flat_idx, m.shape)
print(f"Max position: row {position[0]}, col {position[1]}")

Matrix:
[[38 13 73]
 [10 76  6]
 [80 65 17]
 [ 2 77 72]]

Column sums: [130 231 168]
Row means:    [41.33333333 30.66666667 54.         50.33333333]
Max value:    80
Max position: row 2, col 0


In [23]:
# Exercise 3
heights = np.array([165, 172, 180, 158, 175])
weights = np.array([60, 72, 85, 50, 78])

# Heights in meters
h_m = heights / 100
bmi = weights / h_m**2
print("BMIs:", bmi.round(2))

corr = np.corrcoef(heights, weights)[0, 1]
print(f"\nCorrelation (height vs weight): {corr:.4f}")

BMIs: [22.04 24.34 26.23 20.03 25.47]

Correlation (height vs weight): 0.9988


In [24]:
# Exercise 4
A = np.array([[3, 2, -1],
              [2, -2, 4],
              [-1, 0.5, -1]])
b = np.array([1, -2, 0])

solution = np.linalg.solve(A, b)
print(f"x = {solution[0]:.4f}")
print(f"y = {solution[1]:.4f}")
print(f"z = {solution[2]:.4f}")
print(f"\nCheck: A @ x = {A @ solution}  (should be {b})")

x = 1.0000
y = -2.0000
z = -2.0000

Check: A @ x = [ 1.00000000e+00 -2.00000000e+00 -2.22044605e-16]  (should be [ 1 -2  0])


In [25]:
# Exercise 5
x = np.array([1, 2, 3, 4, 5, 6, 7])
y = np.array([2.1, 4.0, 6.1, 7.9, 10.2, 11.8, 14.1])

# Feature matrix with intercept column
X = np.column_stack([np.ones(len(x)), x])

# Normal equation
w = np.linalg.solve(X.T @ X, X.T @ y)
print(f"Intercept: {w[0]:.4f}")
print(f"Slope:     {w[1]:.4f}")
print("\n(Should be approximately intercept=0, slope=2 based on the data)")

Intercept: 0.0714
Slope:     1.9893

(Should be approximately intercept=0, slope=2 based on the data)
